# Retrieval Augmented Generation (RAG)

RAG (Retrieval Augmented Generation) ist ein Ansatz in der KI, bei dem ein Sprachmodell vor der Antwort relevante Informationen aus externen Wissensquellen abruft und diese in die Generierung der Antwort einbezieht.

**RAG-Pipeline**

1. Dokumente einlesen (PDF / Text)
2. Text in Chunks aufteilen
3. Chunks embedden
4. Embeddings speichern (Vector Store)
5. Frage embedden
6. ähnlichste Chunks finden (Top-K)
7. Prompt mit Kontext bauen
8. Antwort vom LLM erzeugen

👉 Wichtig:
* **Das Modell weiss nichts über die Dokumente.** Alles Wissen kommt ausschliesslich aus dem Kontext.
* Es muss zuerst das Mouser [Magazin Quantum Computing](https://www.mouser.ch/newsroom/publicrelations-methods-quantum-computing-2024final) downgeloaded werden.
* Um wieder von vorne zu Beginnen ist das Verzeichnis "./hf_rag_dataset" zu löschen (`rm -rf ./hf_rag_dataset`).

### Libraries und Konfiguration

Wichtige Punkte:

* Das PDF wird in Text zerlegt und in **überlappende Chunks** aufgeteilt.
* Jeder Chunk erhält ein Embedding **lokal im Notebook** über `sentence-transformers`.
* Das Ergebnis wird als **HF Dataset** lokal gespeichert.
* Für das Retrieval laden wir das Dataset, embedden die Frage lokal und berechnen die **Cosine Similarity** gegen die gespeicherten Chunk-Embeddings.
* Die finale Antwort wird über eine **lokale SGLang-Instanz** via OpenAI-kompatibler Chat API erzeugt.

Benötigte Pakete, falls noch nicht installiert:


In [ ]:
%%bash
source ~/.hf/bin/activate
pip install openai datasets pypdf numpy sentence-transformers einops


In [ ]:
%run ~/work/env.py
! cat ~/work/env.py

In [ ]:
CHAT_MODEL = AI_MODEL
EMBED_MODEL = "nomic-ai/nomic-embed-text-v1.5"

DATASET_DIR = "./hf_rag_dataset"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 6

# Chat läuft über lokale SGLang-Instanz
CHAT_BASE_URL = AI_BASE_URL

# Optional: falls GPU genutzt werden soll, z.B. "cuda"
EMBED_DEVICE = None
EMBED_NORMALIZE = True


---

### Textaufbereitung & Chunking

Vor dem Embedding bereiten wir den Text auf:

* Null-Bytes entfernen
* Whitespace normalisieren
* Text in überlappende Chunks zerlegen

Der Overlap verhindert, dass Sätze oder Abschnitte an Chunk-Grenzen zu hart abgeschnitten werden.

In [ ]:
import re
import hashlib
from typing import List
from pypdf import PdfReader

def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def clean_text(s: str) -> str:
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def chunk_text(text: str) -> List[str]:
    text = clean_text(text)
    chunks = []
    start = 0

    while start < len(text):
        end = min(len(text), start + CHUNK_SIZE)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(0, end - CHUNK_OVERLAP)

    return chunks

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n\n".join(pages)

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

---

### Embeddings lokal im Notebook, Chat über SGLang

Wir trennen klar zwischen:

* **lokalen Embeddings** im Notebook für Retrieval
* **Chat Completion** über die lokale SGLang-Instanz für die finale Antwort

Für `nomic-ai/nomic-embed-text-v1.5` werden Instruktions-Prefixe verwendet:

* `search_document:` für Dokument-Chunks
* `search_query:` für Benutzerfragen

Das verbessert typischerweise die Retrieval-Qualität.


In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import numpy as np

chat_client = OpenAI(
    base_url=CHAT_BASE_URL,
    api_key=OPENAI_API_KEY
)

_embed_model = None

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        kwargs = {"trust_remote_code": True}
        if EMBED_DEVICE:
            kwargs["device"] = EMBED_DEVICE
        _embed_model = SentenceTransformer(EMBED_MODEL, **kwargs)
    return _embed_model

def embed_documents(texts: List[str]) -> List[List[float]]:
    model = get_embed_model()
    prepared = [f"search_document: {text}" for text in texts]
    vectors = model.encode(
        prepared,
        convert_to_numpy=True,
        normalize_embeddings=EMBED_NORMALIZE,
        show_progress_bar=True
    )
    return vectors.astype(np.float32).tolist()

def embed_query(text: str) -> List[float]:
    model = get_embed_model()
    prepared = [f"search_query: {text}"]
    vector = model.encode(
        prepared,
        convert_to_numpy=True,
        normalize_embeddings=EMBED_NORMALIZE,
        show_progress_bar=False
    )
    return vector[0].astype(np.float32).tolist()

def openai_chat(prompt: str) -> str:
    response = chat_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Du beantwortest Fragen ausschliesslich anhand des "
                    "bereitgestellten Kontexts. Wenn der Kontext nicht "
                    "ausreicht, sag das klar."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


---

### Hugging Face Dataset als Speicher

Statt ChromaDB verwenden wir ein **HF Dataset** mit diesen Spalten:

* `id`
* `text`
* `source`
* `chunk`
* `embedding`

Das Dataset wird lokal mit `save_to_disk(...)` gespeichert und bei Bedarf mit `load_from_disk(...)` wieder geladen.

In [ ]:
from datasets import Dataset, load_from_disk

def save_dataset(records, dataset_dir: str = DATASET_DIR):
    ds = Dataset.from_list(records)
    ds.save_to_disk(dataset_dir)
    return ds

def load_dataset(dataset_dir: str = DATASET_DIR):
    return load_from_disk(dataset_dir)

---

### RAG Core – Ingest

Ingest bedeutet hier:

1. PDF oder Textdatei lesen
2. in Chunks zerlegen
3. Chunks lokal embedden
4. alles als HF Dataset speichern

Das LLM beantwortet hier noch nichts. Es wird nur der Retrieval-Bestand aufgebaut.


In [ ]:
def ingest(paths: List[str], dataset_dir: str = DATASET_DIR):
    texts = []
    metas = []

    for path in paths:
        if path.lower().endswith(".pdf"):
            content = read_pdf(path)
        else:
            content = read_text(path)

        chunks = chunk_text(content)

        for i, chunk in enumerate(chunks):
            texts.append(chunk)
            metas.append({
                "id": sha1(f"{path}:{i}:{chunk[:80]}"),
                "source": path,
                "chunk": i,
                "text": chunk
            })

    embeddings = embed_documents(texts)

    records = []
    for meta, emb in zip(metas, embeddings):
        records.append({
            "id": meta["id"],
            "source": meta["source"],
            "chunk": meta["chunk"],
            "text": meta["text"],
            "embedding": emb
        })

    return save_dataset(records, dataset_dir)


---

### Retrieval

Da die Embeddings bereits im HF Dataset liegen, können wir das Retrieval direkt selbst berechnen:

* Frage lokal embedden
* Cosine Similarity gegen alle Chunk-Embeddings berechnen
* Top-K Treffer zurückgeben

Für kleine bis mittlere Dokumentmengen ist das sehr einfach und transparent. Bei sehr grossen Datenmengen würde man später typischerweise wieder einen spezialisierten Index ergänzen.


In [ ]:
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def retrieve(question: str, k: int = TOP_K, dataset_dir: str = DATASET_DIR):
    ds = load_dataset(dataset_dir)
    q_emb = np.array(embed_query(question), dtype=np.float32)

    scored = []
    for row in ds:
        emb = np.array(row["embedding"], dtype=np.float32)
        score = cosine_similarity(q_emb, emb)
        scored.append((score, row))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:k]

    docs = [x[1]["text"] for x in top]
    metas = [{"source": x[1]["source"], "chunk": x[1]["chunk"]} for x in top]
    scores = [x[0] for x in top]
    return docs, metas, scores


Top-K Visualisierung

In [ ]:
def show_top_k(docs, metas, scores, max_chars=500):
    for i, (d, m, score) in enumerate(zip(docs, metas, scores), start=1):
        print(f"\n=== Treffer {i} ===")
        print(f"Quelle : {m['source']}")
        print(f"Chunk  : {m['chunk']}")
        print(f"Score  : {score:.4f}")
        print("-" * 80)
        print(d[:max_chars])

---

### Prompt-Building

Das Modell erhält nicht das ganze PDF, sondern nur die Top-K Chunks als Kontext.

Das ist der eigentliche Kern von RAG: erst Retrieval, dann Generierung.

In [ ]:
def make_prompt(question, docs, metas):
    blocks = []
    for d, m in zip(docs, metas):
        blocks.append(
            f"[Quelle: {m['source']} | Chunk: {m['chunk']}]\n{d}"
        )

    return (
        "Beantworte die Frage nur anhand des Kontexts. "
        "Wenn die Information im Kontext fehlt, sage das ausdrücklich.\n\n"
        "KONTEXT:\n"
        + "\n\n---\n\n".join(blocks)
        + f"\n\nFRAGE:\n{question}\n\nANTWORT:"
    )

---

## Notebook-Use-Case

1. PDF als HF Dataset aufbereiten
2. Top-K Chunks prüfen
3. Antwort über SGLang generieren


In [ ]:
# 1. Index bzw. Dataset bauen
ingest(["engineering-the-quantum-future.pdf"])

In [ ]:
# 2. Frage stellen + Chunks ansehen
docs, metas, scores = retrieve("Explain Quantum Computing", k=5)
show_top_k(docs, metas, scores)

In [ ]:
# 3. Antwort generieren
prompt = make_prompt("Explain Quantum Computing verwende dabei nur die Mouser Informationen und Antworte auf Deutsch", docs, metas)
print(openai_chat(prompt))

---

### Optional: Dataset inspizieren

So kannst du direkt sehen, was im HF Dataset gespeichert wurde.

In [ ]:
ds = load_dataset(DATASET_DIR)
ds